# Theory

The training dataset is the portion of data a model is actually allowed to learn from. During training, the algorithm looks at the features (X_train) alongside the true answers (y_train) and adjusts its internal parameters (coefficients, tree splits, weights, etc.) to minimize prediction error.

# What "Learning" Means, Concretely

For a linear regression model: the training set is used to solve for the coefficients that minimize squared error between predicted and actual Price.

For a random forest: the training set is used to decide which feature and threshold to split on at every node of every tree (e.g., "is Power_BHP > 120?").

The training set is where all of the model's knowledge comes from — nothing the model "knows" can come from outside this set.

# How Big Should It Be?

Bigger training sets generally produce better-generalizing models, up to a point of diminishing returns. This is why the common split favors giving the majority of data to training (e.g., 70–80%), while still reserving enough for honest testing.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = pd.read_csv("used_cars_dataset.csv").drop(columns=["Car_ID"])
X = df.drop(columns=["Price"])
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training rows available to the model:", X_train.shape[0])

numeric_features = ["Age_Years", "Kilometer", "Mileage_kmpl", "Engine_CC", "Power_BHP", "Seats"]
categorical_features = ["Brand", "Model", "Fuel_Type", "Transmission", "Owner_Type", "Color", "City"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
], remainder="passthrough")

model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42))
])

# Model learns ONLY from X_train / y_train
model_pipeline.fit(X_train, y_train)

print("Training R²:", model_pipeline.score(X_train, y_train))

# A Common Misreading of "Training Score"

In [ ]:
train_score = model_pipeline.score(X_train, y_train)
print("Training R²:", train_score)

A very high training score (e.g., 0.98) is not proof of a good model — it may just mean the model has memorized the training rows, including their noise (overfitting). The training score tells you how well the model fits the data it already saw; it says nothing about cars it hasn't seen.